In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Load cleaned dataset
df = pd.read_csv("../Data/Cleaned/ChargingRecords_Cleaned.csv")

# Display first five records
display(df.head())

In [ ]:
# Dataset dimensions
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

# Dataset information
df.info()

In [ ]:
# Demand distribution
plt.figure(figsize=(10, 5))

sns.histplot(
    data=df,
    x="Demand",
    bins=50,
    kde=True
)

plt.title("Distribution of EV Charging Demand")
plt.xlabel("Demand")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
print("Mean:", df["Demand"].mean())
print("Median:", df["Demand"].median())
print("Skewness:", df["Demand"].skew())
print("Kurtosis:", df["Demand"].kurtosis())

In [ ]:
# Calculate Q1 and Q3
Q1 = df["Demand"].quantile(0.25)
Q3 = df["Demand"].quantile(0.75)

# Calculate IQR
IQR = Q3 - Q1

# Calculate lower and upper bounds
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Identify outliers
outliers = df[
    (df["Demand"] < lower_bound) |
    (df["Demand"] > upper_bound)
]

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)
print("Total Outliers:", len(outliers))
print("Outlier Percentage:", round(len(outliers) / len(df) * 100, 2), "%")

In [ ]:
plt.figure(figsize=(10, 4))

sns.boxplot(
    x=df["Demand"]
)

plt.title("Boxplot of EV Charging Demand")
plt.xlabel("Demand")

plt.tight_layout()
plt.show()

In [ ]:
# Analyze Demand by Location

location_demand = (
    df.groupby("Location")["Demand"]
    .mean()
    .sort_values(ascending=False)
)

plt.figure(figsize=(12, 6))

sns.barplot(
    x=location_demand.values,
    y=location_demand.index
)

plt.title("Average EV Charging Demand by Location")
plt.xlabel("Average Demand")
plt.ylabel("Location")

plt.tight_layout()
plt.show()

In [ ]:
# Average Demand by Charger Type

charger_demand = (
    df.groupby("ChargerType")["Demand"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(8, 5))

sns.barplot(
    data=charger_demand,
    x="ChargerType",
    y="Demand"
)

plt.title("Average EV Charging Demand by Charger Type")
plt.xlabel("Charger Type")
plt.ylabel("Average Demand")

plt.tight_layout()
plt.show()

display(charger_demand)

In [ ]:
# Demand by Charger Company

company_demand = (
    df.groupby("ChargerCompany")["Demand"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(8, 5))

sns.barplot(
    data=company_demand,
    x="ChargerCompany",
    y="Demand"
)

plt.title("Average EV Charging Demand by Charger Company")
plt.xlabel("Charger Company")
plt.ylabel("Average Demand")

plt.tight_layout()
plt.show()

display(company_demand)

In [ ]:
# Demand by Day of Week

# Convert StartDay to datetime
df["StartDay"] = pd.to_datetime(df["StartDay"])

# Extract day of week
df["DayOfWeek"] = df["StartDay"].dt.day_name()

# Calculate average demand
day_demand = (
    df.groupby("DayOfWeek")["Demand"]
    .mean()
    .reindex([
        "Monday", "Tuesday", "Wednesday",
        "Thursday", "Friday", "Saturday", "Sunday"
    ])
)

plt.figure(figsize=(10, 5))

sns.barplot(
    x=day_demand.index,
    y=day_demand.values
)

plt.title("Average EV Charging Demand by Day of Week")
plt.xlabel("Day of Week")
plt.ylabel("Average Demand")
plt.xticks(rotation=30)

plt.tight_layout()
plt.show()

display(day_demand)

In [ ]:
# Demand by Hour of Day

# Extract hour from StartTime
df["Hour"] = pd.to_datetime(
    df["StartTime"],
    format="%H:%M:%S"
).dt.hour

# Calculate average demand by hour
hourly_demand = df.groupby("Hour")["Demand"].mean()

# Plot
plt.figure(figsize=(12, 5))

sns.lineplot(
    x=hourly_demand.index,
    y=hourly_demand.values,
    marker="o"
)

plt.title("Average EV Charging Demand by Hour")
plt.xlabel("Hour of Day")
plt.ylabel("Average Demand")
plt.xticks(range(24))
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

display(hourly_demand)

In [ ]:
# Monthly demand analysis

# Extract month from StartDay
df["Month"] = pd.to_datetime(df["StartDay"]).dt.month

# Calculate average demand by month
monthly_demand = df.groupby("Month")["Demand"].mean()

# Plot
plt.figure(figsize=(10, 5))

sns.lineplot(
    x=monthly_demand.index,
    y=monthly_demand.values,
    marker="o"
)

plt.title("Average EV Charging Demand by Month")
plt.xlabel("Month")
plt.ylabel("Average Demand")
plt.xticks(range(1, 13))
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

display(monthly_demand)

In [ ]:
# Monthly charging session counts

monthly_sessions = df.groupby("Month").size()

monthly_summary = pd.DataFrame({
    "Session_Count": monthly_sessions,
    "Average_Demand": df.groupby("Month")["Demand"].mean()
})

display(monthly_summary)

In [ ]:
# Correlation analysis

# Select numerical columns
numerical_columns = [
    "ChargerCompany",
    "ChargerType",
    "Duration",
    "CorrectedDuration",
    "Demand"
]

# Calculate correlation
correlation_matrix = df[numerical_columns].corr()

# Plot heatmap
plt.figure(figsize=(9, 6))

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    linewidths=0.5
)

plt.title("Correlation Matrix of Numerical Features")
plt.tight_layout()
plt.show()

In [ ]:
# Relationship between Duration and Demand

plt.figure(figsize=(10, 5))

sns.scatterplot(
    data=df,
    x="CorrectedDuration",
    y="Demand",
    alpha=0.3
)

plt.title("Relationship Between Charging Duration and Demand")
plt.xlabel("Corrected Duration (Minutes)")
plt.ylabel("Demand")

plt.tight_layout()
plt.show()


In [ ]:
# Analyze Demand by Location and Charger Type

location_charger_demand = (
    df.groupby(["Location", "ChargerType"])["Demand"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(14, 7))

sns.barplot(
    data=location_charger_demand,
    x="Location",
    y="Demand",
    hue="ChargerType"
)

plt.title("Average Demand by Location and Charger Type")
plt.xlabel("Location")
plt.ylabel("Average Demand")
plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.show()

In [ ]:
# Count sessions by location and charger type

location_charger_count = (
    df.groupby(["Location", "ChargerType"])
    .size()
    .reset_index(name="Session_Count")
)

display(
    location_charger_count.sort_values(
        ["Location", "ChargerType"]
    )
)

In [ ]:
# Demand distribution by charger type and location

plt.figure(figsize=(14, 7))

sns.boxplot(
    data=df,
    x="Location",
    y="Demand",
    hue="ChargerType"
)

plt.title("Demand Distribution by Location and Charger Type")
plt.xlabel("Location")
plt.ylabel("Demand")
plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.show()

In [ ]:
# Calculate average demand by weekday and hour

demand_heatmap = df.pivot_table(
    index="DayOfWeek",
    columns="Hour",
    values="Demand",
    aggfunc="mean"
)

# Arrange weekdays
days = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]

demand_heatmap = demand_heatmap.reindex(days)

# Plot heatmap
plt.figure(figsize=(16, 6))

sns.heatmap(
    demand_heatmap,
    cmap="YlOrRd",
    annot=True,
    fmt=".1f"
)

plt.title("Average EV Charging Demand by Day and Hour")
plt.xlabel("Hour of Day")
plt.ylabel("Day of Week")

plt.tight_layout()
plt.show()

In [ ]:
# Count charging sessions by day and hour
session_heatmap = df.pivot_table(
    index="DayOfWeek",
    columns="Hour",
    values="Demand",
    aggfunc="count"
).reindex([
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
])

plt.figure(figsize=(16, 6))

sns.heatmap(
    session_heatmap,
    cmap="Blues",
    fmt=".0f",
    annot=True
)

plt.title("Charging Session Count by Day and Hour")
plt.xlabel("Hour of Day")
plt.ylabel("Day of Week")

plt.tight_layout()
plt.show()